In [ ]:
from google.colab import drive
import os
import pandas as pd

try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")


Mounted at /content/drive
✅ Google Drive mounted successfully!


In [ ]:
!pip install librosa

In [ ]:
import os
import pandas as pd
import numpy as np
import librosa
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, log_loss
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')


In [ ]:
root_folder_path = '/content/drive/MyDrive/ISBInternshipAIML/datasets'

audio_files_data = []
print("Reading folder structure to create dataset metadata...")

# Walk through the directory to get file paths, genres, and languages
for dirpath, dirnames, filenames in os.walk(root_folder_path):
    for filename in [f for f in filenames if f.endswith(".mp3")]:
        file_path = os.path.join(dirpath, filename)

        relative_path = os.path.relpath(dirpath, root_folder_path)
        path_parts = relative_path.split(os.sep)

        genre = path_parts[0] if len(path_parts) > 0 else 'Unknown'
        language = path_parts[1] if len(path_parts) > 1 else 'Unknown'

        audio_files_data.append({
            'path': file_path,
            'genre': genre,
            'language': language,
            'group': os.path.dirname(file_path) # We'll use the folder path for GroupShuffleSplit
        })

df_meta = pd.DataFrame(audio_files_data)
print(f"✅ Found {len(df_meta)} MP3 files.")
print("\n--- Sample of the Metadata ---")
print(df_meta.head())



Reading folder structure to create dataset metadata...
✅ Found 1744 MP3 files.

--- Sample of the Metadata ---
                                                path    genre language  \
0  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
1  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
2  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
3  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
4  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   

                                               group  
0  /content/drive/MyDrive/ISBInternshipAIML/datas...  
1  /content/drive/MyDrive/ISBInternshipAIML/datas...  
2  /content/drive/MyDrive/ISBInternshipAIML/datas...  
3  /content/drive/MyDrive/ISBInternshipAIML/datas...  
4  /content/drive/MyDrive/ISBInternshipAIML/datas...  


In [ ]:
import os
import pandas as pd

root_folder_path = '/content/drive/MyDrive/ISBInternshipAIML/datasets'

audio_files_data = []
print("🔍 Reading folder structure to create dataset metadata...")

for dirpath, dirnames, filenames in os.walk(root_folder_path):
    for filename in [f for f in filenames if f.endswith(".mp3")]:
        file_path = os.path.join(dirpath, filename)

        relative_path = os.path.relpath(dirpath, root_folder_path)
        path_parts = relative_path.split(os.sep)

        genre = path_parts[0] if len(path_parts) > 0 else 'Unknown'
        language = path_parts[1] if len(path_parts) > 1 else 'Unknown'

        audio_files_data.append({
            'path': file_path,
            'genre': genre,
            'language': language,
            'group': os.path.dirname(file_path)
        })

df_meta = pd.DataFrame(audio_files_data)
print(f"✅ Found a total of {len(df_meta)} MP3 files.")


genres_to_keep = [
    'EDM',
    'Folk',
    'Hip-Hop',
    'Pop',
    'film',
    'metal'
]

print(f"\nFiltering dataset to keep only the following genres: {genres_to_keep}")
df_meta = df_meta[df_meta['genre'].isin(genres_to_keep)].copy().reset_index(drop=True)


print(f"Filtered dataset now contains {len(df_meta)} files.")
print("\n--- Sample of the Final Metadata ---")
print(df_meta.head())
print("\n--- Final Genre Distribution ---")
print(df_meta['genre'].value_counts())

🔍 Reading folder structure to create dataset metadata...
✅ Found a total of 1744 MP3 files.

Filtering dataset to keep only the following genres: ['EDM', 'Folk', 'Hip-Hop', 'Pop', 'film', 'metal']
Filtered dataset now contains 1573 files.

--- Sample of the Final Metadata ---
                                                path    genre language  \
0  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
1  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
2  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
3  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   
4  /content/drive/MyDrive/ISBInternshipAIML/datas...  Hip-Hop    Tamil   

                                               group  
0  /content/drive/MyDrive/ISBInternshipAIML/datas...  
1  /content/drive/MyDrive/ISBInternshipAIML/datas...  
2  /content/drive/MyDrive/ISBInternshipAIML/datas...  
3  /content/drive/MyDrive/ISBInternshipAIML/datas...  
4  

In [ ]:

def extract_features(file_path):
    try:
        # Load the audio file
        y, sr = librosa.load(file_path, mono=True, duration=60)

        # Extract various features
        chroma_stft = librosa.feature.chroma_stft(y=y, sr=sr)
        rmse = librosa.feature.rms(y=y)
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(y)
        mfcc = librosa.feature.mfcc(y=y, sr=sr)

        # To ensure every feature vector has the same size, we take the mean
        features = {
            'chroma_stft_mean': np.mean(chroma_stft),
            'rmse_mean': np.mean(rmse),
            'spec_cent_mean': np.mean(spec_cent),
            'spec_bw_mean': np.mean(spec_bw),
            'rolloff_mean': np.mean(rolloff),
            'zcr_mean': np.mean(zcr)
        }
        for i, mfcc_i in enumerate(mfcc):
            features[f'mfcc{i+1}_mean'] = np.mean(mfcc_i)

        return features
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

print("\n Extracting features from audio files... (This may take a while)")

tqdm.pandas()
df_features = df_meta['path'].progress_apply(lambda path: pd.Series(extract_features(path)))

df_full = pd.concat([df_meta, df_features], axis=1).dropna()
print("✅ Feature extraction complete!")
print("\n--- Sample of the Full Dataset with Features ---")
print(df_full.head())



 Extracting features from audio files... (This may take a while)


  3%|▎         | 53/1573 [01:11<33:56,  1.34s/it]


KeyboardInterrupt: 

In [ ]:

from sklearn.model_selection import StratifiedShuffleSplit
le_genre = LabelEncoder()
le_lang = LabelEncoder()

df_full['genre_encoded'] = le_genre.fit_transform(df_full['genre'])
df_full['language_encoded'] = le_lang.fit_transform(df_full['language'])

# Define our features (X) and target variables (y)
X = df_full.drop(columns=['path', 'genre', 'language', 'group', 'genre_encoded', 'language_encoded'])
y_genre = df_full['genre_encoded']
y_lang = df_full['language_encoded']


df_full['stratify_col'] = df_full['genre'] + "_" + df_full['language']

print("🔍 Creating a combined column for stratification...")

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)


train_idx, test_idx = next(sss.split(X, df_full['stratify_col']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_genre_train, y_genre_test = y_genre.iloc[train_idx], y_genre.iloc[test_idx]
y_lang_train, y_lang_test = y_lang.iloc[train_idx], y_lang.iloc[test_idx]

print(f"\n✅ Data split complete using StratifiedShuffleSplit.")
print(f"   Training samples: {len(X_train)}")
print(f"   Testing samples:  {len(X_test)}")

# (Optional) You can verify the distribution is preserved
print("\n--- Verifying Genre Distribution ---")
print("Original:\n", df_full['genre'].value_counts(normalize=True).round(2))
print("\nTraining Set:\n", df_full.iloc[train_idx]['genre'].value_counts(normalize=True).round(2))
print("\nTest Set:\n", df_full.iloc[test_idx]['genre'].value_counts(normalize=True).round(2))


🔍 Creating a combined column for stratification...

✅ Data split complete using StratifiedShuffleSplit.
   Training samples: 1101
   Testing samples:  472

--- Verifying Genre Distribution ---
Original:
 genre
metal      0.34
EDM        0.18
Folk       0.13
Hip-Hop    0.13
Pop        0.12
film       0.11
Name: proportion, dtype: float64

Training Set:
 genre
metal      0.34
EDM        0.18
Folk       0.13
Hip-Hop    0.13
Pop        0.12
film       0.11
Name: proportion, dtype: float64

Test Set:
 genre
metal      0.34
EDM        0.18
Folk       0.13
Hip-Hop    0.12
Pop        0.12
film       0.10
Name: proportion, dtype: float64


In [ ]:

print("\n--- Training Genre Classification Model ---")
rf_genre = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
rf_genre.fit(X_train, y_genre_train)

# Make predictions
y_genre_train_pred = rf_genre.predict(X_train)
y_genre_test_pred = rf_genre.predict(X_test)
y_genre_train_pred_proba = rf_genre.predict_proba(X_train)
y_genre_test_pred_proba = rf_genre.predict_proba(X_test)

# Evaluate the model
train_accuracy_genre = accuracy_score(y_genre_train, y_genre_train_pred)
test_accuracy_genre = accuracy_score(y_genre_test, y_genre_test_pred)
train_loss_genre = log_loss(y_genre_train, y_genre_train_pred_proba)
test_loss_genre = log_loss(y_genre_test, y_genre_test_pred_proba)

print(f" Genre - Training Accuracy: {train_accuracy_genre:.4f}")
print(f" Genre - Training Loss:     {train_loss_genre:.4f}")
print("-" * 35)
print(f" Genre - Testing Accuracy:  {test_accuracy_genre:.4f}")
print(f" Genre - Testing Loss:      {test_loss_genre:.4f}")

# Display detailed classification report
print("\nClassification Report for Genre (Test Set):")
print(classification_report(y_genre_test, y_genre_test_pred, target_names=le_genre.classes_))




--- Training Genre Classification Model ---
 Genre - Training Accuracy: 1.0000
 Genre - Training Loss:     0.2437
-----------------------------------
 Genre - Testing Accuracy:  0.6737
 Genre - Testing Loss:      0.9955

Classification Report for Genre (Test Set):
              precision    recall  f1-score   support

         EDM       0.60      0.58      0.59        84
        Folk       0.55      0.59      0.57        63
     Hip-Hop       0.62      0.61      0.62        59
         Pop       0.57      0.42      0.48        57
        film       0.60      0.51      0.55        49
       metal       0.81      0.92      0.86       160

    accuracy                           0.67       472
   macro avg       0.63      0.61      0.61       472
weighted avg       0.66      0.67      0.67       472



In [ ]:

print("\n--- Training Language Classification Model ---")
rf_lang = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_lang.fit(X_train, y_lang_train)


y_lang_train_pred = rf_lang.predict(X_train)
y_lang_test_pred = rf_lang.predict(X_test)

train_accuracy_lang = accuracy_score(y_lang_train, y_lang_train_pred)
test_accuracy_lang = accuracy_score(y_lang_test, y_lang_test_pred)

y_lang_train_pred_proba = rf_lang.predict_proba(X_train)
y_lang_test_pred_proba = rf_lang.predict_proba(X_test)

train_loss_lang = log_loss(y_lang_train, y_lang_train_pred_proba)
test_loss_lang = log_loss(y_lang_test, y_lang_test_pred_proba)

print(f" Language - Training Accuracy: {train_accuracy_lang:.4f}")
print(f" Language - Testing Accuracy:  {test_accuracy_lang:.4f}")
print("-" * 35)
print(f" Language - Training Loss:     {train_loss_lang:.4f}")
print(f" Language - Testing Loss:      {test_loss_lang:.4f}")

print("\nClassification Report for Language (Test Set):")
print(classification_report(y_lang_test, y_lang_test_pred, target_names=le_lang.classes_))


--- Training Language Classification Model ---
 Language - Training Accuracy: 0.9991
 Language - Testing Accuracy:  0.5614
-----------------------------------
 Language - Training Loss:     0.2804
 Language - Testing Loss:      1.3627

Classification Report for Language (Test Set):
              precision    recall  f1-score   support

     English       0.36      0.55      0.43        60
    Gujarati       0.00      0.00      0.00        13
       Hindi       0.40      0.28      0.33        29
      Korean       0.71      0.43      0.54        23
   Malayalam       0.33      0.07      0.12        14
     Punjabi       1.00      0.30      0.46        10
  Rajasthani       0.86      0.75      0.80         8
     Spanish       0.00      0.00      0.00         8
       Tamil       0.24      0.15      0.18        27
      Telugu       0.34      0.42      0.38        36
     Unknown       0.74      0.96      0.84       160
         eng       0.48      0.41      0.44        61
       hindi 